In [1]:
import sys
print(sys.executable)

c:\Users\MANSI\AppData\Local\Programs\Python\Python311\python.exe


In [2]:
import numpy as np
import pandas as pd
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import os

audio_df = pd.read_csv("improved_features.csv")

text_df = pd.read_csv("temp_text.csv")
text_df = text_df.dropna(subset=["text"])

print("Audio columns:", audio_df.columns.tolist())
print("Text columns:", text_df.columns.tolist())
print("Audio shape:", audio_df.shape)
print("Text shape:", text_df.shape)

Audio columns: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', 'label']
Text columns: ['text', 'label']
Audio shape: (864, 27)
Text shape: (399, 2)


In [6]:
import numpy as np
import pandas as pd
import pickle
from sklearn.metrics import classification_report
import os

# Load models
with open("models/voting_model.pkl", "rb") as f:
    audio_model = pickle.load(f)

with open("models/linguistic_model.pkl", "rb") as f:
    ling_model = pickle.load(f)

with open("models/scaler_new.pkl", "rb") as f:
    scaler = pickle.load(f)

# Audio data
audio_df = pd.read_csv("improved_features.csv")
print("Audio shape:", audio_df.shape)
print("Audio columns:", audio_df.columns.tolist())

# Last column label hai, baki sab features
X_audio = audio_df.iloc[:, :-1].values
y_audio = audio_df.iloc[:, -1].values

print("X_audio shape:", X_audio.shape)
print("Scaler features:", scaler.n_features_in_)

# Scaler features match karo
if X_audio.shape[1] != scaler.n_features_in_:
    X_audio = X_audio[:, :scaler.n_features_in_]
    print("Trimmed to:", X_audio.shape)

X_audio_scaled = scaler.transform(X_audio)

# Audio probabilities
audio_proba = audio_model.predict_proba(X_audio_scaled)
print("Audio proba shape:", audio_proba.shape)

# Text data
text_df = pd.read_csv("temp_text.csv")
text_df = text_df.dropna(subset=["text"])
X_text = text_df["text"].tolist()
y_text = np.array(text_df["label"].tolist())

# Text probabilities
text_proba = ling_model.predict_proba(X_text)
print("Text proba shape:", text_proba.shape)

# Match sizes
min_size = min(len(audio_proba), len(text_proba))
audio_proba = audio_proba[:min_size]
text_proba = text_proba[:min_size]
y_combined = y_audio[:min_size]

# Late fusion
combined_proba = (0.6 * audio_proba) + (0.4 * text_proba)
y_pred = np.argmax(combined_proba, axis=1)

print("\nCombined Model Results:")
print(classification_report(y_combined, y_pred))

# Save
os.makedirs("models", exist_ok=True)
with open("models/combined_model.pkl", "wb") as f:
    pickle.dump({
        'audio_model': audio_model,
        'ling_model': ling_model,
        'scaler': scaler
    }, f)
print("\nCombined model saved!")

Audio shape: (864, 27)
Audio columns: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', 'label']
X_audio shape: (864, 26)
Scaler features: 26
Audio proba shape: (864, 2)
Text proba shape: (399, 2)

Combined Model Results:
              precision    recall  f1-score   support

           0       0.30      1.00      0.47       109
           1       1.00      0.14      0.25       290

    accuracy                           0.38       399
   macro avg       0.65      0.57      0.36       399
weighted avg       0.81      0.38      0.31       399


Combined model saved!
